# Lab 13 - PCA




In [124]:
import sys 
sys.path.append("../")
from utils import *

In [125]:
from scipy.stats import ortho_group
from sklearn.decomposition import PCA
random_state = 5

## Data

### Gaussian data

In [221]:
cov = [3, 0, 0], [0, 1, 0], [0, 0, 0.1]
X = np.random.multivariate_normal([0, 0, 0], cov, size=2500) @ ortho_group.rvs(3, random_state=random_state)

### Circular data

In [222]:
# X = np.random.normal(size=(2, 1000))
# X = np.c_[(10 * X/ np.linalg.norm(X, axis=0)).T, np.random.randn(1000)]
# X = X @ ortho_group.rvs(3)

### Real Data

In [223]:
# df =  pd.read_csv("../data/USArrests.data", index_col=0)
# df = df.drop("UrbanPop", axis=1)

# df = (df - df.mean()) / df.std() 
# X = df.values

## Principal components analysis

Note: the size is defined here as the sqare root of the singular value.

The reason for that definition is to make the size of the plotted principal component to be correlated with the corresponded singular value.
(The longer the axis (principal component) the more variance the data has in that axis)

In [224]:

color_scheme = ["rgb(189,6,96)", "rgb(6,189,99)", "rgb(6,96,189)"]
scene_dict = dict(xaxis=dict(range=[-10, 10]), 
                 yaxis=dict(range=[-10, 10]), 
                 zaxis=dict(range=[-10, 10]))

def get_principal_component(pca, i):
    size = np.sqrt(pca.singular_values_[i-1])/2
    pc = go.Scatter3d(x = [-pca.components_[i-1][0]*size, pca.components_[i-1][0]*size], 
                           y=[-pca.components_[i-1][1]*size, pca.components_[i-1][1]*size], 
                           z=[-pca.components_[i-1][2]*size, pca.components_[i-1][2]*size],
                           marker = dict( size = 1, color = color_scheme[i-1]),
                           line = dict( color = color_scheme[i-1], width = 2*size), name='PC {}'.format(i))
    return pc

In [226]:
pca = PCA(n_components=3).fit(X)

fig = go.Figure(data = [
    go.Scatter3d(x = X[:, 0], y=X[:, 1], z=X[:, 2], opacity = 0.3, mode = 'markers', marker=dict(size=3, color="black")),
    get_principal_component(pca, 1),
    get_principal_component(pca, 2),
    get_principal_component(pca, 3)
]).update_layout(width=800, height=800, title="Original Data", 
                scene = scene_dict).show()

In [228]:
new_components = np.array(pca.transform(pca.components_))
sizes = np.sqrt(pca.singular_values_)

pcs = [go.Scatter3d(x = [-new_components[i][0]*sizes[i], new_components[i][0]*sizes[i]], 
                       y=[-new_components[i][1]*sizes[i], new_components[i][1]*sizes[i]], 
                       z=[-new_components[i][2]*sizes[i], new_components[i][2]*sizes[i]],
                       marker = dict( size = 1, color = "rgb(84,48,5)"),
                       line = dict( color = color_scheme[i], width = 2*sizes[i]))
                        for i in range(3)]

X_transformed = pca.transform(X)
fig = go.Figure(data = [
    go.Scatter3d(x = X_transformed[:, 0], y=X_transformed[:, 1], z=X_transformed[:, 2], 
                 opacity = 0.4, mode = 'markers', marker=dict(size=3, color="black"))] + pcs,
).update_layout(scene=scene_dict, width=800, height=800, title="Transformed Data").show()

### Dimension reduction

PCA reconstruction:


$X \in R^{nXk}, U\in R^{kXk}$
$XU \in R^{nXk}$

$U^{T} XU= \sum_{i=1}^{k}[U^{T}]_{i}\cdot[XU]_{i}$

Note what happens when we zero the last coordinates of X_transform ($XU$):
we are eliminating the last value of the summing - the last principal component "part" of the data. 
This reconstruction of the data reduces the dimension, but keeps the variance maximal as possible (by linear transformation).




In [234]:
X_2d_transformed = X_transformed.copy()
X_2d_transformed[:, 2] = 0
X_2d = pca.inverse_transform(X_2d_transformed)

fig = go.Figure(data = [
    go.Scatter3d(x = X_2d[:, 0], y=X_2d[:, 1], z=X_2d[:, 2], opacity = 0.4, mode = 'markers', marker=dict(size=3, color="black")),
    get_principal_component(pca, 1),
    get_principal_component(pca, 2),
]).update_layout(width=800, height=800, title="2D projection", 
                scene = scene_dict).show()



And there is nothing special in eliminating just the last component:

In [235]:
X_1d_transformed = X_2d_transformed.copy()
X_1d_transformed[:, 1] = 0
X_1d = pca.inverse_transform(X_1d_transformed)


fig = go.Figure(data = [
    go.Scatter3d(x = X_1d[:, 0], y=X_1d[:, 1], z=X_1d[:, 2], opacity = 0.4, mode = 'markers', marker=dict(size=3, color="black")),
    get_principal_component(pca, 1),
]).update_layout(width=800, height=800, title="1D projection", 
                scene = scene_dict).show()


## The singular values and explained variance


We try to choose k big enough to make the lost information — sufficiently small.
we can think of lost informations in terms of variance. If the variance if preserved under the transformation, it's like losing no information. If the new variance is 0, than all data is exactly the same, and all data lost.


The left graph show the singular values - which are the variances along each component.

Note the right graph, and the contribution of each one of the components to the variance that is explained by the components of the transformation. 


Can you spot the original properties of the syntatic data sets that we generated in these graphs?



a really great explanation:

https://ro-che.info/articles/2017-12-11-pca-explained-variance#:~:text=The%20total%20variance%20is%20the,divide%20by%20the%20total%20variance.


In [250]:
# same as pca.singular_values_

# eigenvalues = pca.singular_values_**2
# eigenvalues /= sum(eigenvalues)

cov_mat = np.cov(X.T)
data_variance = np.trace(cov_mat)

fig = make_subplots(rows=1, cols=2, 
specs=[[{'type': 'Bar'}, {'type': 'Waterfall'}]], 
                    subplot_titles=("Singular Values", "Cumulative Explained Variance"))

fig.add_trace(go.Bar(x=['sv1', 'sv2', 'sv3'], y=pca.singular_values_, marker_color = color_scheme), 1, 1)


fig.add_trace(go.Waterfall(
    measure = ["relative", "relative", "relative", "total"],
    x = ["pc1", "pc2", "pc3", "Total"],
    y = list(pca.explained_variance_) + [0]), row=1, col=2
)

fig.add_shape(
    type="rect", fillcolor=color_scheme[0], line=dict(color=color_scheme[0]), opacity=1,
    x0=-0.4, x1=0.4, xref="x", y0=0.0, y1=fig.data[1].y[0], yref="y", row=1, col=2
)

fig.add_shape(
    type="rect", fillcolor=color_scheme[1], line=dict(color=color_scheme[1]), opacity=1,
    x0=0.6, x1=1.4, xref="x", y0=fig.data[1].y[0], y1=fig.data[1].y[0]+fig.data[1].y[1], yref="y", row=1, col=2
)

fig.add_shape(
    type="rect", fillcolor=color_scheme[2], line=dict(color=color_scheme[2]), opacity=1,
    x0=1.6, x1=2.4, xref="x", y0=fig.data[1].y[0]+fig.data[1].y[1], 
    y1=fig.data[1].y[0]+fig.data[1].y[1]+fig.data[1].y[2], yref="y", row=1, col=2
)


fig.add_shape(
    type="rect", fillcolor="Black", line=dict(color="Black"), opacity=1,
    x0=2.6, x1=3.4, xref="x",
    y0=0, y1=data_variance, yref="y", row=1, col=2
)




fig.update_layout(showlegend=False)

fig.show()

